In [5]:
import pandas as pd

sec_df = pd.read_csv(
    "/lakehouse/default/Files/data/processed/securities/security_processed.csv"
)

sec_df = sec_df[
    ["isin", "symbol"]
].drop_duplicates()

print("Rows:", len(sec_df))

sec_df.head()

StatementMeta(, 0fbbaf7f-9b04-46c4-9ace-723a19d8c226, 10, Finished, Available, Finished, False)

Rows: 4964


,isin,symbol
0,INF204KB17R6,08ABB
1,INF204KB11S7,08ADD
2,INF204KB12S5,08ADR
3,INF204KB18R4,08AGG
4,INF204KB13S3,08AMD


In [6]:
print("Rows:", len(sec_df))

print(
    "Unique ISIN:",
    sec_df["isin"].nunique()
)

print(
    "Unique Symbol:",
    sec_df["symbol"].nunique()
)

StatementMeta(, 0fbbaf7f-9b04-46c4-9ace-723a19d8c226, 12, Finished, Available, Finished, False)

Rows: 4964
Unique ISIN: 4963
Unique Symbol: 4963


In [7]:
sec_df = sec_df.drop_duplicates(
    subset=["isin"]
)

print("Rows:", len(sec_df))

print(
    "Unique ISIN:",
    sec_df["isin"].nunique()
)

StatementMeta(, 0fbbaf7f-9b04-46c4-9ace-723a19d8c226, 13, Finished, Available, Finished, False)

Rows: 4963
Unique ISIN: 4963


In [8]:
# testing on some companies
import yfinance as yf
import pandas as pd

sample_symbols = [
    "RELIANCE",
    "TCS",
    "INFY",
    "HDFCBANK",
    "ICICIBANK"
]

results = []

for symbol in sample_symbols:

    try:

        info = yf.Ticker(
            f"{symbol}.NS"
        ).info

        results.append({
            "symbol": symbol,
            "sector": info.get("sector"),
            "industry": info.get("industry")
        })

    except Exception:

        results.append({
            "symbol": symbol,
            "sector": None,
            "industry": None
        })

pd.DataFrame(results)

StatementMeta(, 0fbbaf7f-9b04-46c4-9ace-723a19d8c226, 14, Finished, Available, Finished, False)

,symbol,sector,industry
0,RELIANCE,Energy,Oil & Gas Refining & Marketing
1,TCS,Technology,Information Technology Services
2,INFY,Technology,Information Technology Services
3,HDFCBANK,Financial Services,Banks - Regional
4,ICICIBANK,Financial Services,Banks - Regional


In [9]:
import yfinance as yf
import pandas as pd

sample_df = sec_df.head(100)

results = []

for _, row in sample_df.iterrows():

    symbol = row["symbol"]

    try:

        info = yf.Ticker(
            f"{symbol}.NS"
        ).info

        results.append({
            "isin": row["isin"],
            "symbol": symbol,
            "sector": info.get("sector"),
            "industry": info.get("industry")
        })

    except Exception:

        results.append({
            "isin": row["isin"],
            "symbol": symbol,
            "sector": None,
            "industry": None
        })

lookup_df = pd.DataFrame(results)

print("Rows:", len(lookup_df))

print(
    "Sector Found:",
    lookup_df["sector"].notna().sum()
)

print(
    "Industry Found:",
    lookup_df["industry"].notna().sum()
)

lookup_df.head()

StatementMeta(, 0fbbaf7f-9b04-46c4-9ace-723a19d8c226, 16, Finished, Available, Finished, False)

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 08ABB.NS"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 08ADD.NS"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 08ADR.NS"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 08AGG.NS"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 08AMD.NS"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 08AMR.NS"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 08AQD.NS"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","descrip

Rows: 100
Sector Found: 37
Industry Found: 37


,isin,symbol,sector,industry
0,INF204KB17R6,08ABB,None,None
1,INF204KB11S7,08ADD,None,None
2,INF204KB12S5,08ADR,None,None
3,INF204KB18R4,08AGG,None,None
4,INF204KB13S3,08AMD,None,None


In [10]:
nse_mcap = pd.read_csv(
    "/lakehouse/default/Files/data/raw/nse_mcap/mcap05062026.csv"
)

nse_symbols = set(
    nse_mcap["Symbol"]
    .astype(str)
    .str.strip()
)

sec_df["in_nse_mcap"] = sec_df["symbol"].isin(
    nse_symbols
)

print(
    "Total securities:",
    len(sec_df)
)

print(
    "Found in NSE MCAP:",
    sec_df["in_nse_mcap"].sum()
)

StatementMeta(, 0fbbaf7f-9b04-46c4-9ace-723a19d8c226, 19, Finished, Available, Finished, False)

Total securities: 4963
Found in NSE MCAP: 2348


database that will be sent to yahoo

In [12]:
nse_lookup_df = sec_df[
    sec_df["symbol"].isin(nse_symbols)
].copy()

print(
    "Companies to fetch:",
    len(nse_lookup_df)
)

nse_lookup_df.head()

StatementMeta(, 0fbbaf7f-9b04-46c4-9ace-723a19d8c226, 22, Finished, Available, Finished, False)

Companies to fetch: 2348


,isin,symbol,in_nse_mcap
31,INE144J01027,20MICRONS,True
33,INE253B01015,21STCENMGM,True
35,INE466L01038,360ONE,True
37,INE994E01018,3BBLACKBIO,True
41,INE748C01038,3IINFOLTD,True


In [13]:
import yfinance as yf
import pandas as pd

from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed
)

save_path = (
    "/lakehouse/default/Files/data/processed/"
    "securities/sector_industry_lookup.csv"
)


def fetch_sector_industry(row):

    symbol = row["symbol"]

    try:

        info = yf.Ticker(
            f"{symbol}.NS"
        ).info

        return {
            "isin": row["isin"],
            "symbol": symbol,
            "sector": info.get("sector"),
            "industry": info.get("industry")
        }

    except Exception:

        return {
            "isin": row["isin"],
            "symbol": symbol,
            "sector": None,
            "industry": None
        }


results = []

with ThreadPoolExecutor(
    max_workers=20
) as executor:

    futures = [
        executor.submit(
            fetch_sector_industry,
            row
        )

        for _, row in nse_lookup_df.iterrows()
    ]

    for i, future in enumerate(
        as_completed(futures),
        start=1
    ):

        results.append(
            future.result()
        )

        if i % 100 == 0:

            pd.DataFrame(
                results
            ).to_csv(
                save_path,
                index=False
            )

            print(
                f"Saved progress: {i}"
            )

lookup_df = pd.DataFrame(
    results
)

lookup_df.to_csv(
    save_path,
    index=False
)

print("SUCCESS")

print(
    "Rows:",
    len(lookup_df)
)

print(
    "Sector Found:",
    lookup_df["sector"].notna().sum()
)

print(
    "Industry Found:",
    lookup_df["industry"].notna().sum()
)

StatementMeta(, 0fbbaf7f-9b04-46c4-9ace-723a19d8c226, 24, Finished, Available, Finished, False)

Saved progress: 100
Saved progress: 200
Saved progress: 300
Saved progress: 400
Saved progress: 500
Saved progress: 600
Saved progress: 700
Saved progress: 800
Saved progress: 900
Saved progress: 1000
Saved progress: 1100
Saved progress: 1200
Saved progress: 1300
Saved progress: 1400
Saved progress: 2000
Saved progress: 2100
Saved progress: 2200
Saved progress: 2300
SUCCESS
Rows: 2348
Sector Found: 1543
Industry Found: 1543


HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"User is unable to access this feature - https://bit.ly/yahoo-finance-api-feedback"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"User is unable to access this feature - https://bit.ly/yahoo-finance-api-feedback"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"User is unable to access this feature - https://bit.ly/yahoo-finance-api-feedback"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"User is unable to access this feature - https://bit.ly/yahoo-finance-api-feedback"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"User is unable to access this feature - https://bit.ly/yahoo-finance-api-feedback"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"Invalid Crumb"}}}
HTTP Error 401: {"financ

In [ ]:
lookup_df = pd.read_csv(
    "/lakehouse/default/Files/data/processed/securities/sector_industry_lookup.csv"
)

print("Rows:", len(lookup_df))

print(
    "Sector Found:",
    lookup_df["sector"].notna().sum()
)

print(
    "Industry Found:",
    lookup_df["industry"].notna().sum()
)

print(
    "Unique Sectors:",
    lookup_df["sector"].nunique()
)

print(
    "Unique Industries:",
    lookup_df["industry"].nunique()
)

In [3]:
import pandas as pd
lookup_df = pd.read_csv(
    "/lakehouse/default/Files/data/processed/securities/sector_industry_lookup.csv"
)

print(
    lookup_df["sector"]
    .value_counts()
    .head(20)
)

print()

print(
    lookup_df["industry"]
    .value_counts()
    .head(20)
)

StatementMeta(, 8b4c04b8-290c-4c3e-9758-7802ee8f5d3c, 6, Finished, Available, Finished, False)

sector
Industrials               310
Consumer Cyclical         271
Basic Materials           242
Financial Services        172
Healthcare                115
Consumer Defensive        112
Technology                111
Real Estate                56
Communication Services     45
Utilities                  30
Energy                     28
Name: count, dtype: int64

industry
Specialty Chemicals                         70
Drug Manufacturers - Specialty & Generic    67
Textile Manufacturing                       61
Engineering & Construction                  60
Auto Parts                                  55
Specialty Industrial Machinery              51
Capital Markets                             43
Packaged Foods                              43
Steel                                       43
Credit Services                             43
Information Technology Services             42
Real Estate - Development                   39
Agricultural Inputs                         35
Electrical Equip

In [4]:
failed_df = lookup_df[
    lookup_df["sector"].isna()
]

print(
    "Failed:",
    len(failed_df)
)

StatementMeta(, 8b4c04b8-290c-4c3e-9758-7802ee8f5d3c, 7, Finished, Available, Finished, False)

Failed: 856
